# ✦ Buffy AI — Chatbot ស្ទីល GPT នៅលើ Google Colab

កម្មវិធីជជែក AI ដែល៖
- 💬 ឆ្លើយសំណួរបានគ្រប់យ៉ាង ជាភាសាខ្មែរ ឬភាសាណាមួយ
- 🌐 ស្វែងរកអ៊ីនធឺណិត (Google Search) ដូច ChatGPT ព្រមទាំងបង្ហាញប្រភព
- 🖼️ ផ្ញើរូបភាពឱ្យ AI មើល និងវិភាគ
- 🎨 បង្កើត / កែរូបភាពដោយ AI (Nano Banana)
- 🔗 បង្កើត **link សាធារណៈ** សម្រាប់ជជែកពីទូរស័ព្ទ ឬកុំព្យូទ័រណាមួយ

---

## ជំហានទី 1 — យក API key ឥតគិតថ្លៃ
1. បើក 👉 https://aistudio.google.com/apikey
2. ចូលដោយគណនី Google រួចចុច **Create API key**
3. ចម្លង key (វាមានទម្រង់បែប `AIza...`)

## ជំហានទី 2 — រត់ cell ទី 1 និងទី 2
- Cell **1️⃣ ដំឡើង** — ដំឡើងបណ្ណាល័យ (រត់ម្តងប៉ុណ្ណោះ)
- Cell **2️⃣ ដាក់ API key** — paste key របស់អ្នក (លាក់មិនបង្ហាញ)

> គន្លឹះ៖ បើអ្នកមិនចង់ paste រាល់ដងទេ សូមដាក់ key ក្នុង 🔑 (Secrets) ខាងឆ្វេងដៃ ជាឈ្មោះ `GEMINI_API_KEY` ហើយបើក “Notebook access”។

## ជំហានទី 3 — រត់ cell ចុងក្រោយ
វានឹងបង្ហាញ **link** បែប `https://xxxx.trycloudflare.com` — ចម្លងទៅបើកក្នុង browser ហើយចាប់ផ្តើមជជែកបាន។

⏳ Link នេះដំណើរការដរាបណា cell ចុងក្រោយនៅរត់ (Colab បិទពេលអ្នកបិទ tab យូរ ឬអស់ម៉ោង)។
🔒 កុំចែករំលែក link ជាសាធារណៈ — អ្នកណាមាន link អាចប្រើ quota API key របស់អ្នកបាន។

In [ ]:
#@title 1️⃣ ដំឡើងបណ្ណាល័យ (រត់ម្តងប៉ុណ្ណោះ)
%pip install -q --upgrade fastapi uvicorn google-genai

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
%%writefile ui.py
# -*- coding: utf-8 -*-
"""Shared single-page chat UI (ChatGPT-style) used by app.py and local_app.py.

It is plain HTML/CSS/JS with no build step: Markdown rendering, code blocks with
copy buttons, image attachments, source chips, chat export and local history.
"""

PAGE = r"""<!doctype html>
<html lang="km">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1, maximum-scale=1" />
<title>Buffy AI</title>
<link rel="preconnect" href="https://fonts.googleapis.com" />
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin />
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=Noto+Sans+Khmer:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet" />
<script src="https://cdn.jsdelivr.net/npm/marked@12.0.2/marked.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/dompurify@3.1.6/dist/purify.min.js"></script>
<style>
  :root{
    --bg:#0f1012; --panel:#17181b; --panel-2:#1d1f23; --border:#2a2d33;
    --text:#ececf1; --muted:#9aa0aa; --accent:#10a37f; --accent-2:#7c5cff;
    --user:#2b2f36; --code:#0b0c0e; --radius:16px;
  }
  *{box-sizing:border-box}
  html,body{height:100%}
  body{
    margin:0; background:var(--bg); color:var(--text);
    font-family:"Inter","Noto Sans Khmer",system-ui,-apple-system,"Segoe UI",sans-serif;
    font-size:15px; line-height:1.65; -webkit-font-smoothing:antialiased;
  }
  button{font-family:inherit}
  .app{display:flex; height:100dvh; overflow:hidden}

  /* sidebar */
  .sidebar{
    width:272px; flex:0 0 272px; background:var(--panel); border-right:1px solid var(--border);
    display:flex; flex-direction:column; padding:14px; gap:12px;
  }
  .brand{display:flex; align-items:center; gap:10px; padding:6px 6px 2px}
  .brand .logo{width:32px;height:32px;border-radius:10px;background:linear-gradient(135deg,var(--accent),var(--accent-2));display:grid;place-items:center;font-size:17px}
  .brand h1{font-size:16px;margin:0;font-weight:700;letter-spacing:.2px}
  .brand small{display:block;color:var(--muted);font-size:11px;font-weight:500;line-height:1.2}
  .new-chat{
    display:flex;align-items:center;justify-content:center;gap:8px;width:100%;padding:11px 12px;
    border-radius:12px;border:1px solid var(--border);background:var(--panel-2);color:var(--text);
    font-size:14px;font-weight:600;cursor:pointer;transition:.15s;
  }
  .new-chat:hover{background:#24272d;border-color:#3a3e46}
  .side-label{font-size:11px;text-transform:uppercase;letter-spacing:.9px;color:var(--muted);padding:6px 6px 0;font-weight:600}
  .field{display:flex;flex-direction:column;gap:6px;padding:0 4px}
  .field label{font-size:12.5px;color:var(--muted);font-weight:500}
  select{
    width:100%;padding:9px 10px;border-radius:10px;background:var(--panel-2);color:var(--text);
    border:1px solid var(--border);font-size:13.5px;outline:none;cursor:pointer;
  }
  select:focus{border-color:var(--accent)}
  .toggle{display:flex;align-items:center;justify-content:space-between;gap:10px;padding:8px 10px;border-radius:10px;background:var(--panel-2);border:1px solid var(--border);cursor:pointer}
  .toggle span{font-size:13px;display:flex;align-items:center;gap:8px}
  .toggle input{display:none}
  .switch{width:36px;height:20px;border-radius:20px;background:#3a3e46;position:relative;transition:.18s;flex:0 0 auto}
  .switch::after{content:"";position:absolute;top:2px;left:2px;width:16px;height:16px;border-radius:50%;background:#fff;transition:.18s}
  .toggle input:checked + .switch{background:var(--accent)}
  .toggle input:checked + .switch::after{transform:translateX(16px)}
  .side-actions{display:flex;gap:8px;padding:0 4px}
  .ghost{
    flex:1;padding:9px 8px;border-radius:10px;background:transparent;border:1px solid var(--border);
    color:var(--muted);font-size:12.5px;cursor:pointer;transition:.15s;
  }
  .ghost:hover{color:var(--text);border-color:#3a3e46;background:var(--panel-2)}
  .side-foot{margin-top:auto;font-size:11px;color:#6f7580;padding:6px;line-height:1.5}

  /* main */
  .main{flex:1;display:flex;flex-direction:column;min-width:0;position:relative;background:var(--bg)}
  .topbar{
    display:flex;align-items:center;gap:10px;padding:10px 16px;border-bottom:1px solid var(--border);
    background:rgba(15,16,18,.85);backdrop-filter:blur(8px);z-index:5;
  }
  .icon-btn{
    width:36px;height:36px;border-radius:10px;display:grid;place-items:center;border:1px solid transparent;
    background:transparent;color:var(--muted);cursor:pointer;font-size:16px;
  }
  .icon-btn:hover{background:var(--panel-2);color:var(--text)}
  #menuBtn{display:none}
  .top-title{font-size:14px;font-weight:600;flex:1;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
  .pill{font-size:11.5px;color:var(--muted);border:1px solid var(--border);padding:3px 9px;border-radius:999px;background:var(--panel-2);white-space:nowrap}

  .chat{flex:1;overflow-y:auto;scroll-behavior:smooth;padding:22px 16px 8px}
  .thread{max-width:820px;margin:0 auto;display:flex;flex-direction:column;gap:22px}

  /* empty state */
  .empty{max-width:720px;margin:6vh auto 0;text-align:center;padding:0 8px}
  .empty .big{
    width:62px;height:62px;border-radius:20px;margin:0 auto 18px;display:grid;place-items:center;font-size:29px;
    background:linear-gradient(135deg,var(--accent),var(--accent-2));box-shadow:0 14px 40px rgba(124,92,255,.28)
  }
  .empty h2{font-size:23px;margin:0 0 8px;font-weight:700}
  .empty p{color:var(--muted);margin:0 0 26px;font-size:14px}
  .chips{display:flex;flex-wrap:wrap;gap:10px;justify-content:center}
  .chip{
    padding:10px 14px;border-radius:12px;border:1px solid var(--border);background:var(--panel);
    color:var(--text);font-size:13px;cursor:pointer;transition:.15s;text-align:left;
  }
  .chip:hover{border-color:var(--accent);background:var(--panel-2);transform:translateY(-1px)}

  /* messages */
  .msg{display:flex;gap:12px;align-items:flex-start}
  .msg .avatar{
    width:30px;height:30px;border-radius:9px;flex:0 0 auto;display:grid;place-items:center;font-size:15px;margin-top:2px;
    background:linear-gradient(135deg,var(--accent),var(--accent-2));color:#fff
  }
  .msg .body{min-width:0;flex:1}
  .msg.user{justify-content:flex-end}
  .msg.user .body{flex:0 1 auto;max-width:82%}
  .msg.user .bubble{background:var(--user);padding:11px 15px;border-radius:16px 16px 4px 16px;white-space:pre-wrap;word-wrap:break-word}
  .msg.user .thumbs{display:flex;gap:8px;flex-wrap:wrap;justify-content:flex-end;margin-bottom:8px}
  .msg.user .thumbs img{width:104px;height:104px;object-fit:cover;border-radius:12px;border:1px solid var(--border)}

  .md{word-wrap:break-word;overflow-wrap:anywhere}
  .md > *:first-child{margin-top:0}
  .md > *:last-child{margin-bottom:0}
  .md h1,.md h2,.md h3,.md h4{margin:1.3em 0 .55em;line-height:1.3;font-weight:700}
  .md h1{font-size:1.42em}.md h2{font-size:1.26em}.md h3{font-size:1.1em}.md h4{font-size:1em}
  .md p{margin:.65em 0}
  .md ul,.md ol{margin:.6em 0;padding-left:1.5em}
  .md li{margin:.28em 0}
  .md a{color:#7fb2ff;text-decoration:none}
  .md a:hover{text-decoration:underline}
  .md blockquote{margin:.7em 0;padding:.4em 0 .4em 1em;border-left:3px solid var(--accent);color:#c8ccd4}
  .md table{border-collapse:collapse;width:100%;margin:.8em 0;font-size:13.5px;display:block;overflow-x:auto}
  .md th,.md td{border:1px solid var(--border);padding:7px 10px;text-align:left}
  .md th{background:var(--panel-2);font-weight:600}
  .md code{
    background:rgba(255,255,255,.08);padding:.15em .4em;border-radius:5px;font-size:.88em;
    font-family:"JetBrains Mono",ui-monospace,SFMono-Regular,Menlo,monospace;
  }
  .md pre{position:relative;background:var(--code);border:1px solid var(--border);border-radius:12px;padding:14px 14px 12px;overflow-x:auto;margin:.9em 0}
  .md pre code{background:none;padding:0;font-size:12.8px;line-height:1.6}
  .copy-code{
    position:absolute;top:8px;right:8px;font-size:11px;padding:4px 9px;border-radius:7px;cursor:pointer;
    background:#22252b;border:1px solid var(--border);color:var(--muted);opacity:0;transition:.15s;
  }
  .md pre:hover .copy-code{opacity:1}
  .copy-code:hover{color:var(--text)}

  .msg-actions{display:flex;gap:6px;margin-top:8px;opacity:0;transition:.15s}
  .msg:hover .msg-actions{opacity:1}
  .msg-actions button{background:transparent;border:1px solid transparent;color:var(--muted);font-size:11.5px;padding:4px 8px;border-radius:7px;cursor:pointer}
  .msg-actions button:hover{background:var(--panel-2);color:var(--text);border-color:var(--border)}

  .status{display:flex;align-items:center;gap:8px;color:var(--muted);font-size:12.5px;margin-bottom:6px}
  .status .spinner{width:13px;height:13px;border-radius:50%;border:2px solid #3a3e46;border-top-color:var(--accent);animation:spin .8s linear infinite}
  @keyframes spin{to{transform:rotate(360deg)}}
  .blink{display:inline-block;width:7px;height:15px;background:var(--accent);border-radius:2px;margin-left:3px;vertical-align:-2px;animation:blink 1s steps(2) infinite}
  @keyframes blink{50%{opacity:0}}
  .sources{margin-top:12px;border-top:1px solid var(--border);padding-top:10px}
  .sources .lbl{font-size:11.5px;color:var(--muted);margin-bottom:7px;font-weight:600;letter-spacing:.3px}
  .source-list{display:flex;flex-wrap:wrap;gap:8px}
  .source{
    display:inline-flex;align-items:center;gap:6px;max-width:260px;padding:6px 11px;border-radius:999px;
    background:var(--panel-2);border:1px solid var(--border);color:#cfd3da;font-size:12px;text-decoration:none;
  }
  .source:hover{border-color:var(--accent);color:#fff}
  .source b{font-weight:600;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
  .gen-img{margin-top:10px}
  .gen-img img{max-width:100%;border-radius:14px;border:1px solid var(--border);display:block}
  .gen-img a{display:inline-block;margin-top:8px;font-size:12px;color:#7fb2ff;text-decoration:none}

  /* composer */
  .composer-wrap{padding:10px 16px 16px;background:linear-gradient(to top,var(--bg) 55%,rgba(15,16,18,0))}
  .composer{max-width:820px;margin:0 auto}
  .attachments{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:8px}
  .attachments .att{position:relative}
  .attachments img{width:62px;height:62px;object-fit:cover;border-radius:10px;border:1px solid var(--border)}
  .attachments .rm{position:absolute;top:-6px;right:-6px;width:19px;height:19px;border-radius:50%;border:none;background:#3a3e46;color:#fff;font-size:11px;cursor:pointer;display:grid;place-items:center}
  .box{
    display:flex;align-items:flex-end;gap:8px;background:var(--panel-2);border:1px solid var(--border);
    border-radius:18px;padding:8px 8px 8px 12px;transition:.15s;box-shadow:0 8px 28px rgba(0,0,0,.28)
  }
  .box:focus-within{border-color:#3f8f79}
  .box.gen:focus-within{border-color:var(--accent-2)}
  #input{
    flex:1;resize:none;border:none;outline:none;background:transparent;color:var(--text);
    font-family:inherit;font-size:15px;line-height:1.55;max-height:190px;padding:7px 2px;
  }
  #input::placeholder{color:#767c86}
  .send{
    width:38px;height:38px;flex:0 0 auto;border-radius:12px;border:none;cursor:pointer;font-size:16px;
    background:var(--accent);color:#06241c;display:grid;place-items:center;transition:.15s;
  }
  .send:hover:not(:disabled){filter:brightness(1.12)}
  .send:disabled{opacity:.45;cursor:not-allowed}
  .send.gen{background:var(--accent-2);color:#fff}
  .send.stop{background:#c0392b;color:#fff}
  .hint{max-width:820px;margin:8px auto 0;text-align:center;font-size:11px;color:#6f7580}

  @media (max-width:820px){
    #menuBtn{display:grid}
    .sidebar{
      position:fixed;inset:0 auto 0 0;z-index:40;transform:translateX(-100%);transition:transform .22s ease;
      box-shadow:0 0 40px rgba(0,0,0,.55);height:100dvh
    }
    .sidebar.open{transform:none}
    .scrim{position:fixed;inset:0;background:rgba(0,0,0,.5);z-index:35;display:none}
    .scrim.show{display:block}
    .msg.user .body{max-width:92%}
    .chat{padding:16px 10px 4px}
    .top-title{display:none}
  }
</style>
</head>
<body>
<div class="app">
  <aside class="sidebar" id="sidebar">
    <div class="brand">
      <div class="logo">✦</div>
      <div>
        <h1>Buffy AI</h1>
        <small>ជំនួយការ AI ឆ្លាតវៃ</small>
      </div>
    </div>
    <button class="new-chat" id="newChat">＋ ការសន្ទនាថ្មី</button>

    <div class="side-label">ការកំណត់</div>
    <div class="field">
      <label for="model">ម៉ូដែល</label>
      <select id="model">
        <option value="gemini-3.8-flash" selected>Gemini 3.8 Flash · ឆ្លាតបំផុត</option>
        <option value="gemini-3.6-flash">Gemini 3.6 Flash</option>
        <option value="gemini-3.5-flash-lite">Gemini 3.5 Flash-Lite · លឿន</option>
      </select>
    </div>
    <label class="toggle" for="webSearch">
      <span>🌐 ស្វែងរកអ៊ីនធឺណិត</span>
      <input type="checkbox" id="webSearch" checked />
      <i class="switch"></i>
    </label>
    <label class="toggle" for="imageMode">
      <span>🎨 បង្កើតរូបភាព</span>
      <input type="checkbox" id="imageMode" />
      <i class="switch"></i>
    </label>

    <div class="side-actions">
      <button class="ghost" id="downloadChat">⬇️ ទាញយក</button>
      <button class="ghost" id="clearChat">🗑 លាង</button>
    </div>

    <div class="side-foot">
      ប្រើប្រាស់ Gemini API · ប្រវត្តិរក្សាទុកក្នុងកម្មវិធីរុករករបស់អ្នក
    </div>
  </aside>
  <div class="scrim" id="scrim"></div>

  <main class="main">
    <div class="topbar">
      <button class="icon-btn" id="menuBtn" aria-label="menu">☰</button>
      <div class="top-title" id="topTitle">ការសន្ទនាថ្មី</div>
      <span class="pill" id="modePill">💬 ជជែក</span>
    </div>

    <div class="chat" id="chat">
      <div class="thread" id="thread">
        <div class="empty" id="empty">
          <div class="big">✦</div>
          <h2>តើខ្ញុំអាចជួយអ្វីបានខ្លះ?</h2>
          <p>សួរបានគ្រប់សំណួរ · ផ្ញើរូបភាពឱ្យខ្ញុំមើល · ឬបើក “បង្កើតរូបភាព”</p>
          <div class="chips">
            <button class="chip">ពន្យល់អំពី AI និងរបៀបដែលវាដំណើរការ</button>
            <button class="chip">សរសេរកូដ Python សម្រាប់គណនាផ្ទៃក្រឡា</button>
            <button class="chip">តើមានព័ត៌មានថ្មីៗអ្វីខ្លះថ្ងៃនេះ?</button>
            <button class="chip">បង្កើតរូបភាពឆ្មាញ៉ាំកាហ្វេ ស្ទីលគំនូរទឹក</button>
          </div>
        </div>
      </div>
    </div>

    <div class="composer-wrap">
      <div class="composer">
        <div class="attachments" id="attachments"></div>
        <div class="box" id="box">
          <button class="icon-btn" id="attachBtn" title="ភ្ជាប់រូបភាព">📎</button>
          <input type="file" id="fileInput" accept="image/*" multiple hidden />
          <textarea id="input" rows="1" placeholder="សួរអ្វីក៏បាន… (Enter ដើម្បីផ្ញើ)"></textarea>
          <button class="send" id="sendBtn" title="ផ្ញើ">➤</button>
        </div>
        <div class="hint">AI អាចធ្វើខុសបាន · សូមផ្ទៀងផ្ទាត់ព័ត៌មានសំខាន់ៗ</div>
      </div>
    </div>
  </main>
</div>

<script>
(function () {
  "use strict";
  const $ = (s) => document.querySelector(s);
  const thread = $("#thread");
  const emptyEl = $("#empty");
  const chatEl = $("#chat");
  const input = $("#input");
  const sendBtn = $("#sendBtn");
  const box = $("#box");
  const attEl = $("#attachments");
  const fileInput = $("#fileInput");
  const modelSel = $("#model");
  const webSearch = $("#webSearch");
  const imageMode = $("#imageMode");
  const modePill = $("#modePill");
  const topTitle = $("#topTitle");
  const STORE = "buffy-ai-chat-v2";

  const state = { messages: [], attachments: [], busy: false, controller: null };

  marked.setOptions({ breaks: true, gfm: true });
  const md = (text) => DOMPurify.sanitize(marked.parse(text || ""), { ADD_ATTR: ["target"] });

  function esc(s) {
    return String(s == null ? "" : s).replace(/[&<>"']/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;" }[c]));
  }

  /* ---------------- persistence ---------------- */
  function save() {
    const data = JSON.stringify({
      messages: state.messages.map((m) => ({ role: m.role, text: m.text, sources: m.sources || [], img: m.img || null, images: m.images || [] })),
      model: modelSel.value,
      web: webSearch.checked,
    });
    try { localStorage.setItem(STORE, data); } catch (e) { /* quota, ignore */ }
  }
  function load() {
    try {
      const raw = localStorage.getItem(STORE);
      if (!raw) return;
      const d = JSON.parse(raw);
      if (Array.isArray(d.messages) && d.messages.length) {
        state.messages = d.messages;
        if (d.model) modelSel.value = d.model;
        if (typeof d.web === "boolean") webSearch.checked = d.web;
      }
    } catch (e) { /* ignore */ }
  }

  /* ---------------- rendering ---------------- */
  function hideEmpty() { if (emptyEl && emptyEl.parentNode) emptyEl.remove(); }

  function newBubble(role, msg) {
    hideEmpty();
    const wrap = document.createElement("div");
    wrap.className = "msg " + role;
    let thumbs = "";
    if (role === "user" && msg && msg.images && msg.images.length) {
      thumbs = '<div class="thumbs">' + msg.images.map((im) => '<img src="' + esc(im.url) + '" alt="image" />').join("") + "</div>";
    }
    const avatar = role === "assistant" ? '<div class="avatar">✦</div>' : "";
    const inner = role === "user"
      ? '<div class="body">' + thumbs + '<div class="bubble">' + esc(msg ? msg.text : "") + "</div></div>"
      : '<div class="avatar">✦</div><div class="body"><div class="md"></div><div class="status" hidden><i class="spinner"></i><span>កំពុងគិត…</span></div><div class="cfg"></div></div>';
    wrap.innerHTML = inner;
    thread.appendChild(wrap);
    return wrap;
  }

  function renderAssistantBody(node, m) {
    const body = node.querySelector(".md");
    body.innerHTML = md(m.text || "") + (state.busy && !m.done ? '<i class="blink"></i>' : "");
    body.querySelectorAll("pre").forEach((pre) => {
      if (pre.querySelector(".copy-code")) return;
      const b = document.createElement("button");
      b.className = "copy-code";
      b.textContent = "copy";
      b.onclick = () => {
        const code = pre.querySelector("code");
        navigator.clipboard.writeText(code ? code.innerText : pre.innerText).then(() => {
          b.textContent = "copied ✓";
          setTimeout(() => (b.textContent = "copy"), 1200);
        });
      };
      pre.appendChild(b);
    });
    const cfg = node.querySelector(".cfg");
    let html = "";
    if (m.img) {
      html += '<div class="gen-img"><img src="' + esc(m.img.url) + '" alt="generated" /><a href="' + esc(m.img.url) + '" download="buffy-ai.png">⬇️ ទាញយករូបភាព</a></div>';
    }
    if (m.sources && m.sources.length) {
      html += '<div class="sources"><div class="lbl">ប្រភព</div><div class="source-list">' +
        m.sources.map((s) => {
          let host = s.url;
          try { host = new URL(s.url).hostname.replace(/^www\./, ""); } catch (e) {}
          return '<a class="source" href="' + esc(s.url) + '" target="_blank" rel="noopener"><span>🔗</span><b>' + esc(host) + "</b></a>";
        }).join("") + "</div></div>";
    }
    if (m.text && m.done) {
      html += '<div class="msg-actions"><button data-act="copy">⧉ ចម្លង</button></div>';
    }
    cfg.innerHTML = html;
    const copyBtn = cfg.querySelector('[data-act="copy"]');
    if (copyBtn) copyBtn.onclick = () => navigator.clipboard.writeText(m.text || "");
  }

  function setStatus(node, text) {
    const st = node.querySelector(".status");
    if (!st) return;
    if (!text) { st.hidden = true; return; }
    st.hidden = false;
    st.querySelector("span").textContent = text;
  }

  function scrollToBottom(force) {
    const nearBottom = chatEl.scrollHeight - chatEl.scrollTop - chatEl.clientHeight < 180;
    if (force || nearBottom) chatEl.scrollTop = chatEl.scrollHeight;
  }

  function renderAll() {
    thread.innerHTML = "";
    state.messages.forEach((m) => {
      const node = newBubble(m.role, m);
      if (m.role === "assistant") {
        m.done = true;
        renderAssistantBody(node, m);
      }
    });
    scrollToBottom(true);
    if (!state.messages.length) {
      // rebuild empty state
      const e = document.createElement("div");
      e.className = "empty";
      e.id = "empty";
      e.innerHTML = '<div class="big">✦</div><h2>តើខ្ញុំអាចជួយអ្វីបានខ្លះ?</h2>' +
        '<p>សួរបានគ្រប់សំណួរ · ផ្ញើរូបភាពឱ្យខ្ញុំមើល · ឬបើក “បង្កើតរូបភាព”</p>' +
        '<div class="chips">' +
        '<button class="chip">ពន្យល់អំពី AI និងរបៀបដែលវាដំណើរការ</button>' +
        '<button class="chip">សរសេរកូដ Python សម្រាប់គណនាផ្ទៃក្រឡា</button>' +
        '<button class="chip">តើមានព័ត៌មានថ្មីៗអ្វីខ្លះថ្ងៃនេះ?</button>' +
        '<button class="chip">បង្កើតរូបភាពឆ្មាញ៉ាំកាហ្វេ ស្ទីលគំនូរទឹក</button>' +
        "</div>";
      thread.appendChild(e);
      bindChips(e);
    }
  }

  function bindChips(root) {
    (root || document).querySelectorAll(".chip").forEach((c) => {
      c.onclick = () => { input.value = c.textContent; send(); };
    });
  }

  /* ---------------- attachments ---------------- */
  function readFile(file) {
    return new Promise((resolve) => {
      const fr = new FileReader();
      fr.onload = () => {
        const url = fr.result;
        if (file.size < 380 * 1024) return resolve({ data: url.split(",")[1], mime: file.type || "image/jpeg", url });
        const img = new Image();
        img.onload = () => {
          const max = 1280;
          let w = img.width, h = img.height;
          const scale = Math.min(1, max / Math.max(w, h));
          w = Math.round(w * scale); h = Math.round(h * scale);
          const cv = document.createElement("canvas");
          cv.width = w; cv.height = h;
          cv.getContext("2d").drawImage(img, 0, 0, w, h);
          const out = cv.toDataURL("image/jpeg", 0.86);
          resolve({ data: out.split(",")[1], mime: "image/jpeg", url: out });
        };
        img.onerror = () => resolve({ data: url.split(",")[1], mime: file.type || "image/jpeg", url });
        img.src = url;
      };
      fr.readAsDataURL(file);
    });
  }

  function renderAttachments() {
    attEl.innerHTML = state.attachments.map((a, i) =>
      '<div class="att"><img src="' + esc(a.url) + '" alt="" /><button class="rm" data-i="' + i + '">✕</button></div>'
    ).join("");
    attEl.querySelectorAll(".rm").forEach((b) => {
      b.onclick = () => { state.attachments.splice(+b.dataset.i, 1); renderAttachments(); };
    });
  }

  fileInput.onchange = async () => {
    const files = [...fileInput.files].slice(0, 4);
    for (const f of files) {
      if (!f.type.startsWith("image/")) continue;
      if (state.attachments.length >= 4) break;
      state.attachments.push(await readFile(f));
    }
    fileInput.value = "";
    renderAttachments();
  };

  /* ---------------- send / stream ---------------- */
  function setBusy(on) {
    state.busy = on;
    sendBtn.disabled = false;
    sendBtn.classList.toggle("stop", on);
    sendBtn.textContent = on ? "■" : "➤";
    box.classList.toggle("gen", imageMode.checked && !on);
  }

  function historyPayload() {
    return state.messages
      .filter((m) => m.text && (m.role === "user" || m.role === "assistant"))
      .slice(-24)
      .map((m) => ({ role: m.role === "assistant" ? "model" : "user", text: m.text }));
  }

  async function send() {
    if (state.busy) return;
    const text = input.value.trim();
    if (!text && !state.attachments.length) return;

    const images = state.attachments.map((a) => ({ data: a.data, mime_type: a.mime }));
    const previews = state.attachments.map((a) => ({ url: a.url }));
    const userMsg = { role: "user", text, images: previews };
    state.messages.push(userMsg);
    const userNode = newBubble("user", userMsg);
    input.value = "";
    autoGrow();
    state.attachments = [];
    renderAttachments();
    scrollToBottom(true);

    const aMsg = { role: "assistant", text: "", sources: [], img: null, done: false };
    state.messages.push(aMsg);
    const aNode = newBubble("assistant");
    setStatus(aNode, "កំពុងតភ្ជាប់…");
    setBusy(true);

    const controller = new AbortController();
    state.controller = controller;
    let raf = null;
    const paint = () => {
      if (raf) return;
      raf = requestAnimationFrame(() => { raf = null; renderAssistantBody(aNode, aMsg); scrollToBottom(false); });
    };

    try {
      const res = await fetch("/api/chat", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({
          message: text,
          model: modelSel.value,
          web_search: !!webSearch.checked,
          image_mode: !!imageMode.checked,
          images,
          history: historyPayload().slice(0, -1),
        }),
        signal: controller.signal,
      });
      if (!res.ok || !res.body) throw new Error("HTTP " + res.status + " " + (await res.text()).slice(0, 200));

      const reader = res.body.getReader();
      const dec = new TextDecoder();
      let buf = "";
      let firstToken = true;

      while (true) {
        const { value, done } = await reader.read();
        if (done) break;
        buf += dec.decode(value, { stream: true });
        let idx;
        while ((idx = buf.indexOf("\n\n")) >= 0) {
          const raw = buf.slice(0, idx).trim();
          buf = buf.slice(idx + 2);
          if (!raw.startsWith("data:")) continue;
          let evt;
          try { evt = JSON.parse(raw.slice(5).trim()); } catch (e) { continue; }
          if (evt.type === "text") {
            if (firstToken) { firstToken = false; setStatus(aNode, null); }
            aMsg.text += evt.text;
            paint();
          } else if (evt.type === "status") {
            setStatus(aNode, evt.text);
          } else if (evt.type === "sources") {
            aMsg.sources = evt.sources || [];
            paint();
          } else if (evt.type === "image") {
            aMsg.img = { url: "data:" + (evt.mime || "image/png") + ";base64," + evt.data };
            paint();
          } else if (evt.type === "error") {
            aMsg.text += (aMsg.text ? "\n\n" : "") + "> ⚠️ " + evt.text;
            paint();
          } else if (evt.type === "done") {
            aMsg.done = true;
          }
        }
      }
    } catch (err) {
      if (err && err.name === "AbortError") {
        aMsg.text += "\n\n_(បានបញ្ឈប់)_";
      } else {
        aMsg.text += (aMsg.text ? "\n\n" : "") + "> ⚠️ " + (err.message || err);
      }
    } finally {
      aMsg.done = true;
      setStatus(aNode, null);
      setBusy(false);
      state.controller = null;
      renderAssistantBody(aNode, aMsg);
      scrollToBottom(false);
      save();
      updateTitle();
    }
  }

  /* ---------------- controls ---------------- */
  function autoGrow() {
    input.style.height = "auto";
    input.style.height = Math.min(input.scrollHeight, 190) + "px";
  }
  function updateMode() {
    modePill.textContent = imageMode.checked ? "🎨 បង្កើតរូបភាព" : (webSearch.checked ? "🌐 ស្វែងរក + ជជែក" : "💬 ជជែក");
    input.placeholder = imageMode.checked ? "ពិពណ៌នារូបភាពដែលអ្នកចង់បាន…" : "សួរអ្វីក៏បាន… (Enter ដើម្បីផ្ញើ)";
    box.classList.toggle("gen", imageMode.checked);
  }
  function updateTitle() {
    const first = state.messages.find((m) => m.role === "user");
    topTitle.textContent = first ? first.text.slice(0, 60) : "ការសន្ទនាថ្មី";
  }

  input.addEventListener("input", autoGrow);
  input.addEventListener("keydown", (e) => {
    if (e.key === "Enter" && !e.shiftKey) { e.preventDefault(); send(); }
  });
  sendBtn.onclick = () => {
    if (state.busy) {
      if (state.controller) state.controller.abort();
      return;
    }
    send();
  };
  $("#attachBtn").onclick = () => fileInput.click();
  webSearch.onchange = () => { updateMode(); save(); };
  imageMode.onchange = () => { updateMode(); save(); };
  modelSel.onchange = () => save();

  $("#newChat").onclick = () => {
    if (state.busy && state.controller) state.controller.abort();
    state.messages = [];
    state.attachments = [];
    renderAttachments();
    renderAll();
    save();
    updateTitle();
    closeSidebar();
  };
  $("#clearChat").onclick = () => { $("#newChat").onclick(); };

  $("#downloadChat").onclick = () => {
    if (!state.messages.length) return;
    let out = "# Buffy AI — ប្រវត្តិការសន្ទនា\n\n";
    state.messages.forEach((m) => {
      out += (m.role === "user" ? "## 🧑 អ្នក\n\n" : "## ✦ Buffy AI\n\n") + (m.text || "") + "\n\n";
      if (m.sources && m.sources.length) {
        out += m.sources.map((s) => "- [" + s.title + "](" + s.url + ")").join("\n") + "\n\n";
      }
    });
    const blob = new Blob([out], { type: "text/markdown;charset=utf-8" });
    const a = document.createElement("a");
    a.href = URL.createObjectURL(blob);
    a.download = "buffy-ai-chat.md";
    a.click();
    setTimeout(() => URL.revokeObjectURL(a.href), 4000);
  };

  /* mobile sidebar */
  function closeSidebar() { $("#sidebar").classList.remove("open"); $("#scrim").classList.remove("show"); }
  $("#menuBtn").onclick = () => { $("#sidebar").classList.add("open"); $("#scrim").classList.add("show"); };
  $("#scrim").onclick = closeSidebar;

  /* boot */
  load();
  bindChips(document);
  updateMode();
  renderAll();
  updateTitle();
  autoGrow();
  input.focus();
})();
</script>
</body>
</html>
"""


In [ ]:
%%writefile app.py
# -*- coding: utf-8 -*-
"""Buffy AI — a GPT-style chat app powered by the Gemini API.

Features
- Streaming chat replies (server-sent events)
- Google Search grounding with clickable sources (like ChatGPT web search)
- Image upload / understanding, and image generation & editing (Nano Banana)
- Beautiful dark UI, Markdown + code blocks, chat export, local history

Run locally:  GEMINI_API_KEY=... uvicorn app:app --host 127.0.0.1 --port 8000
In Colab:     the notebook writes this file and opens a public cloudflared link.
"""

import base64
import json
import os
import traceback

from fastapi import FastAPI
from fastapi.responses import HTMLResponse, StreamingResponse
from pydantic import BaseModel
from typing import List, Optional

from google import genai

from ui import PAGE

APP_NAME = "Buffy AI"

# Chat models offered in the UI. The allow-list also protects the public link
# from being used with arbitrary (expensive) model names.
CHAT_MODELS = {
    "gemini-3.8-flash": "Gemini 3.8 Flash",
    "gemini-3.6-flash": "Gemini 3.6 Flash",
    "gemini-3.5-flash-lite": "Gemini 3.5 Flash-Lite",
}
DEFAULT_MODEL = "gemini-3.8-flash"
IMAGE_MODEL = "gemini-nano-banana-2.1"

SYSTEM_INSTRUCTION = (
    "You are a helpful, highly knowledgeable AI assistant. "
    "Always answer in the same language the user writes in (Khmer, English, or any other). "
    "Answer every question completely and accurately, with structure: use Markdown headings, "
    "lists, tables and fenced code blocks with language tags where useful. "
    "When web search results are provided, base current facts on them and cite naturally. "
    "If you are not sure about something, say so briefly and give the best available answer. "
    "Be warm and clear, never rude, and never refuse a reasonable question."
)

app = FastAPI(title=APP_NAME)
_client = None


def get_client() -> "genai.Client":
    global _client
    if _client is None:
        key = os.environ.get("GEMINI_API_KEY", "").strip()
        if not key:
            raise RuntimeError(
                "GEMINI_API_KEY is not set. Add it in the Colab cell / environment first."
            )
        _client = genai.Client(api_key=key)
    return _client


# --------------------------------------------------------------------------- #
# Request models
# --------------------------------------------------------------------------- #
class ImageIn(BaseModel):
    data: str = ""
    mime_type: str = "image/png"


class Turn(BaseModel):
    role: str = "user"
    text: str = ""


class ChatRequest(BaseModel):
    message: str = ""
    model: str = DEFAULT_MODEL
    web_search: bool = True
    image_mode: bool = False
    images: List[ImageIn] = []
    history: List[Turn] = []


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def sse(obj: dict) -> str:
    return "data: " + json.dumps(obj, ensure_ascii=False) + "\n\n"


def clean_b64(data: str) -> str:
    if not data:
        return ""
    if "," in data and data.strip().startswith("data:"):
        return data.split(",", 1)[1]
    return data.strip()


def build_turn(message: str, images: List[ImageIn]) -> dict:
    parts = []
    if message.strip():
        parts.append({"type": "text", "text": message.strip()})
    for img in images[:4]:
        raw = clean_b64(img.data)
        if not raw:
            continue
        parts.append(
            {
                "type": "image",
                "data": raw,
                "mime_type": img.mime_type or "image/png",
            }
        )
    if not parts:
        parts.append({"type": "text", "text": "(empty message)"})
    return {"type": "user_input", "content": parts}


def build_history_steps(history: List[Turn]) -> List[dict]:
    steps = []
    for turn in history[-30:]:
        text = (turn.text or "").strip()
        if not text:
            continue
        if turn.role == "user":
            steps.append({"type": "user_input", "content": [{"type": "text", "text": text}]})
        else:
            steps.append({"type": "model_output", "content": [{"type": "text", "text": text}]})
    return steps


def collect_sources(interaction) -> List[dict]:
    sources, seen = [], set()
    for step in getattr(interaction, "steps", None) or []:
        if getattr(step, "type", None) != "model_output":
            continue
        for block in getattr(step, "content", None) or []:
            for ann in getattr(block, "annotations", None) or []:
                if getattr(ann, "type", None) != "url_citation":
                    continue
                url = getattr(ann, "url", None)
                if not url or url in seen:
                    continue
                seen.add(url)
                sources.append({"title": getattr(ann, "title", None) or url, "url": url})
    return sources[:8]


def friendly_error(exc: Exception) -> str:
    msg = str(exc) or exc.__class__.__name__
    low = msg.lower()
    if "api key" in low or "api_key" in low or "permission" in low or "401" in low or "403" in low:
        return "API key មិនត្រឹមត្រូវ ឬគ្មានសិទ្ធិ។ សូមពិនិត្យ GEMINI_API_KEY ឡើងវិញ។"
    if "quota" in low or "429" in low or "rate" in low:
        return "ប្រើប្រាស់លើសកំណត់ (rate limit) — សូមរង់ចាំមួយសន្ទុះ ហើយព្យាយាមម្ដងទៀត។"
    return "មានបញ្ហាបច្ចេកទេស៖ " + msg[:300]


# --------------------------------------------------------------------------- #
# Streaming chat
# --------------------------------------------------------------------------- #
def image_stream(req: ChatRequest):
    """Non-streaming image generation / editing (Nano Banana)."""
    try:
        yield sse({"type": "status", "text": "កំពុងបង្កើតរូបភាព…"})
        steps = build_history_steps(req.history) + [build_turn(req.message, req.images)]
        interaction = get_client().interactions.create(
            model=IMAGE_MODEL,
            input=steps,
        )
        img = getattr(interaction, "output_image", None)
        data = getattr(img, "data", None) if img is not None else None
        mime = getattr(img, "mime_type", None) if img is not None else None
        if data:
            yield sse({"type": "image", "data": data, "mime": mime or "image/png"})
        text = getattr(interaction, "output_text", None)
        if text:
            yield sse({"type": "text", "text": text})
        if not data and not text:
            yield sse({"type": "error", "text": "មិនអាចបង្កើតរូបភាពបានទេ។ សូមសាកល្បងបញ្ជាផ្សេង។"})
        yield sse({"type": "done"})
    except Exception as exc:  # noqa: BLE001
        traceback.print_exc()
        yield sse({"type": "error", "text": friendly_error(exc)})


def chat_stream(req: ChatRequest):
    streamed = False
    try:
        model = req.model if req.model in CHAT_MODELS else DEFAULT_MODEL
        steps = build_history_steps(req.history) + [build_turn(req.message, req.images)]
        kwargs = {
            "model": model,
            "input": steps,
            "system_instruction": SYSTEM_INSTRUCTION,
            "stream": True,
        }
        if req.web_search:
            kwargs["tools"] = [{"type": "google_search"}]

        stream = get_client().interactions.create(**kwargs)
        for event in stream:
            etype = getattr(event, "event_type", None)

            if etype == "step.delta":
                delta = getattr(event, "delta", None)
                dtype = getattr(delta, "type", None)
                if dtype == "text":
                    chunk = getattr(delta, "text", "") or ""
                    if chunk:
                        streamed = True
                        yield sse({"type": "text", "text": chunk})
                elif dtype == "thought_summary":
                    yield sse({"type": "status", "text": "កំពុងគិត…"})
                elif dtype == "google_search_call":
                    queries = ""
                    args = getattr(delta, "arguments", None)
                    if isinstance(args, dict):
                        queries = ", ".join(args.get("queries") or [])
                    yield sse(
                        {
                            "type": "status",
                            "text": "កំពុងស្វែងរកអ៊ីនធឺណិត…" + (" " + queries if queries else ""),
                        }
                    )

            elif etype == "interaction.completed":
                interaction = getattr(event, "interaction", None)
                if interaction is not None:
                    sources = collect_sources(interaction)
                    if sources:
                        yield sse({"type": "sources", "sources": sources})
                    if not streamed:
                        text = getattr(interaction, "output_text", None)
                        if text:
                            streamed = True
                            yield sse({"type": "text", "text": text})

        if not streamed:
            yield sse({"type": "error", "text": "មិនមានចម្លើយត្រឡប់មកវិញទេ។ សូមព្យាយាមម្ដងទៀត។"})
        yield sse({"type": "done"})
    except Exception as exc:  # noqa: BLE001
        traceback.print_exc()
        yield sse({"type": "error", "text": friendly_error(exc)})


@app.post("/api/chat")
def chat(req: ChatRequest):
    generator = image_stream(req) if req.image_mode else chat_stream(req)
    return StreamingResponse(
        generator,
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache, no-transform",
            "X-Accel-Buffering": "no",
            "Connection": "keep-alive",
        },
    )


@app.get("/api/health")
def health():
    return {
        "ok": True,
        "app": APP_NAME,
        "key_set": bool(os.environ.get("GEMINI_API_KEY", "").strip()),
        "models": list(CHAT_MODELS.keys()),
        "image_model": IMAGE_MODEL,
    }


# --------------------------------------------------------------------------- #
# Front-end (single page, no build step)
# --------------------------------------------------------------------------- #


@app.get("/", response_class=HTMLResponse)
def index():
    return HTMLResponse(PAGE)


In [ ]:
#@title 2️⃣ ដាក់ API key របស់អ្នក
import os

key = ""
try:  # ព្យាយាមអានពី Colab Secrets (🔑) មុន
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY") or ""
except Exception:
    key = ""

if not key:
    from getpass import getpass
    key = getpass("Paste your Gemini API key (https://aistudio.google.com/apikey): ").strip()

os.environ["GEMINI_API_KEY"] = key
print("✅ ដាក់ API key រួចរាល់" if key else "❌ មិនមាន key — សូមរត់ cell នេះម្តងទៀត")

In [ ]:
#@title 3️⃣ បើក server + យក link សាធារណៈ
import re, subprocess, threading, time, urllib.request

import uvicorn

PORT = 8000

threading.Thread(
    target=uvicorn.run,
    args=("app:app",),
    kwargs=dict(host="127.0.0.1", port=PORT, log_level="warning"),
    daemon=True,
).start()

ready = False
for _ in range(80):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=2) as r:
            print("✅ Server ដំណើរការ:", r.read().decode()[:160])
            ready = True
            break
    except Exception:
        time.sleep(0.5)
if not ready:
    raise RuntimeError("Server មិនអាចចាប់ផ្តើមបានទេ — សូមមើល log ខាងលើ")

print("🌐 កំពុងបើក tunnel… (រង់ចាំ 5-15 វិនាទី)\n")

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

found = False
for line in proc.stdout:
    if not found:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            found = True
            print("=" * 68)
            print("🎉 នេះជា link សម្រាប់ជជែក (បើកបានគ្រប់ឧបករណ៍):\n")
            print("   " + m.group(0))
            print("\n" + "=" * 68)
            print("⚠️ កុំបិទ cell នេះ — កម្មវិធីដំណើរការពេល cell នេះនៅរត់។")
            print("   ចុច ⏹ ដើម្បីបញ្ឈប់។")

print("\n🔌 Tunnel បានបិទ។ ប្រសិនបើអ្នកចង់បាន link ថ្មី សូមរត់ cell នេះម្តងទៀត។")